In [8]:
import pandas as pd
import sys
sys.path.append('../../')
from MEGA_utilities import data_col_standardize, data_remove_duplicate


# load data
Crick_H1N1 = pd.read_excel('../../../data/raw/data4model(Crick-H1N1).xlsx')
Crick_H3N2 = pd.read_excel('../../../data/raw/data4model(Crick-H3N2).xlsx')
origin_df = pd.concat([Crick_H1N1, Crick_H3N2]).reset_index(drop=True)

In [12]:
origin_df.columns

Index(['serumName', 'serumPassage', 'serumPassCat', 'serumDate', 'serumType',
       'ferret', 'virusName', 'virusPassage', 'virusPassCat', 'virusDate',
       'virusType', 'dataSource', 'serumIslID', 'serumMatchedPass',
       'virusIslID', 'virusMatchedPass', 'HI_Dist', 'serumHA', 'serumNA',
       'virusHA', 'virusNA'],
      dtype='object')

## DNA MEGA

In [34]:
## select required columns
AA_data_filt1 = origin_df[['serumName','virusName','serumHA', 'serumNA', 'virusHA', 'virusNA', 
                           'serumPassCat','virusPassCat', 'serumType','HI_Dist']].copy()
## remove duplicated row and mean HI_Dist
AA_data_filt2 = AA_data_filt1.groupby(['serumHA', 'serumNA', 'virusHA', 'virusNA', 'serumPassCat', 'virusPassCat']) \
        .agg({'serumName': 'first', 'virusName': 'first', 'serumType': 'first', 'HI_Dist': 'mean'}) \
        .reset_index()[['serumName', 'virusName', 'serumHA', 'serumNA', 'virusHA', 'virusNA',
                        'serumPassCat', 'virusPassCat', 'serumType', 'HI_Dist']]
## remove PassCat = 'BOTH'
AA_data_filt3 = AA_data_filt2[(AA_data_filt2['serumPassCat'] != 'BOTH') &
                              (AA_data_filt2['virusPassCat'] != 'BOTH')].reset_index(drop=True)
## replace PassCat to special token
AA_data_filt4 = AA_data_filt3.replace({'serumPassCat': {'EGG': '<EGG>', 'CELL': '<CELL>'},
                                       'virusPassCat': {'EGG': '<EGG>', 'CELL': '<CELL>'}})

In [35]:
import torch
from torch.utils.data import Dataset, DataLoader

class AADataset(Dataset):
    def __init__(self, DataFrame):
        self.sequence = (DataFrame['serumHA'] + '<eos>' + DataFrame['serumNA'] + '<eos>' + DataFrame['virusHA'] + \
                         '<eos>' + DataFrame['virusNA'] + '<eos>' + DataFrame['serumType'] + \
                         '<eos>' + DataFrame['serumPassCat'] + '<eos>' + DataFrame['virusPassCat']).tolist()
        self.labels = torch.tensor(DataFrame['HI_Dist'].tolist())
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return self.sequence[idx], self.labels[idx]

In [36]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(AA_data_filt4, test_size=0.1, random_state=42)
train_df, valid_df = train_test_split(train_df, test_size=1/9, random_state=42)

train_dataset = AADataset(train_df)
valid_dataset = AADataset(valid_df)
test_dataset = AADataset(test_df)

train_loader = DataLoader(train_dataset, batch_size=10, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=128, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

In [12]:
# train_df.to_csv('../../../data/processed/1.1/AA_train_df.csv', index=True)
# valid_df.to_csv('../../../data/processed/1.1/AA_valid_df.csv', index=True)
# test_df.to_csv('../../../data/processed/1.1/AA_test_df.csv', index=True)

In [39]:
from bio_tokenizer import BioTokenizer
from transformers import MegaConfig, MegaForSequenceClassification
from MEGA_utilities import count_parameters
from torch.optim import AdamW
from transformers import get_scheduler
import torch

# get tokenizer and model
tokenizer = BioTokenizer(vocab_file='./vocab_AA.txt')

# update the num_vocab and num_label
config = MegaConfig()
config.num_labels=1
config.vocab_size=28
config.max_positions=4000
config.num_attention_heads=4
config.num_hidden_layers=5
device = torch.device("cuda:0")
model = MegaForSequenceClassification(config)
model.to(device)
print("Number of parameters: %e"%count_parameters(model))

# optimizer
optimizer = AdamW(model.parameters(), lr=5e-4)

# scheduler
num_epochs = 160
num_training_steps = num_epochs * len(train_loader)
lr_scheduler = get_scheduler(name="linear", optimizer=optimizer,
                             num_warmup_steps=len(train_loader), num_training_steps=num_training_steps)

Number of parameters: 1.135435e+06


In [40]:
from tqdm import tqdm
from utilities import print_exams
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.stats import pearsonr, spearmanr
from utilities import EarlyStopping
from datetime import datetime

progress_bar = tqdm(range(num_training_steps))
early_stopping = EarlyStopping(patience=10, delta=0.005, save_dir='./1.2_HANA_model/')

# Training loop
for epoch in range(num_epochs):
    model.train()
    loss_ls = []
    for batch_seq, batch_label in train_loader:
        batch_input = tokenizer(batch_seq, padding='longest', return_tensors="pt")
        batch_input = batch_input.to(device)
        batch_label = batch_label.to(device)

        outputs = model(**batch_input, labels=batch_label)

        loss = outputs.loss
        loss_ls.append(loss.item())

        loss.backward()
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)
    train_loss = sum(loss_ls) / len(loss_ls)
    print('train loss :', train_loss)

    prediction_ls = []
    reference_ls = []
    logits_ls = []
    loss_ls_valid = []
    with torch.no_grad():
        model.eval()
        for batch_seq, batch_label in valid_loader:
            batch_input = tokenizer(batch_seq, padding='longest', return_tensors="pt")
            batch_input = batch_input.to(device)
            batch_label = batch_label.to(device)

            outputs = model(**batch_input, labels=batch_label)
            logits = outputs.logits
            loss = outputs.loss

            logits_ls.append(logits)
            loss_ls_valid.append(loss.item())
            prediction_ls += logits.tolist()
            prediction_ls_final = []
            for sublist in prediction_ls:
                for element in sublist:
                    prediction_ls_final.append(element)
            reference_ls += batch_label.tolist()

    print_exams(prediction_ls_final, reference_ls)
    valid_MAE = mean_absolute_error(reference_ls, prediction_ls_final)
    valid_mse = mean_squared_error(reference_ls, prediction_ls_final)
    valid_pearson = pearsonr(reference_ls, prediction_ls_final).statistic
    valid_spearman = spearmanr(reference_ls, prediction_ls_final).statistic
    
    early_stopping(valid_mse, model)
    if early_stopping.early_stop:
        print("Early stopping")
        break

    ## 将epoch信息写入log.txt
    with open('./1.2_HANA_model/log.txt', 'a') as f:
        current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        f.write(f"[{current_time}] Epoch {epoch + 1}/{num_epochs}, train loss: {train_loss:.4f}, valid MAE: {valid_MAE:.4f}, valid MSE: {valid_mse:.4f}, valid Pearson: {valid_pearson:.4f}, valid Spearman: {valid_spearman:.4f}\n")

  1%|          | 5760/921760 [10:48<28:23:39,  8.96it/s]

train loss : 3.066953901216359
MAE:  1.2963774632954825
MSE:  2.9344842906678146
pearson correlation:  PearsonRResult(statistic=0.4809142512890819, pvalue=0.0)
spearman correlation:  SignificanceResult(statistic=0.5135782271982356, pvalue=0.0)
Validation MSE decrease (inf --> 2.934484).  Saving model ...


  1%|          | 11521/921760 [22:41<28:08:46,  8.98it/s] 

train loss : 2.7744951199877907


  1%|▏         | 11523/921760 [23:44<3704:23:31, 14.65s/it]

...

 88%|████████▊ | 812302/921760 [27:49:59<453:50:31, 14.93s/it]

MAE:  0.6665481759335884
MSE:  0.7568531895013159
pearson correlation:  PearsonRResult(statistic=0.8949792484865275, pvalue=0.0)
spearman correlation:  SignificanceResult(statistic=0.8695099348401291, pvalue=0.0)
EarlyStopping counter: 9 out of 10


 89%|████████▉ | 818062/921760 [28:00:43<3:07:08,  9.24it/s]  

train loss : 0.5319448062799893
MAE:  0.6685947792803996
MSE:  0.7615920201405453
pearson correlation:  PearsonRResult(statistic=0.8956326898056348, pvalue=0.0)
spearman correlation:  SignificanceResult(statistic=0.8697912879887458, pvalue=0.0)
EarlyStopping counter: 10 out of 10
Early stopping


: 